<a href="https://colab.research.google.com/github/UnfoldDataScience/Agentic_Ai_For_Beginner/blob/main/Part3/MAS_Using_MSAutogen_Part_3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026(Tested on Google Colab)

# ⚠️ Important: If running in Colab , Switch to Python 3.12

Before running any other cells in this notebook, please switch your Google Colab runtime to **Python 3.12**.

From the top right arrow (next to Connect/RAM), select:

**Change runtime type → Runtime Version → 2026.07**

The next cell should output python version as 3.12


In [ ]:
!python --version

Python 3.12.13


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   GROQ_API_KEY

In [ ]:
import warnings
warnings.filterwarnings('ignore')

💡 Note: While running package installation command below,Google Colab may display some dependency messages during installation. These relate to packages we are not using in this notebook, so you can safely continue when the installation completes.

In [ ]:
!pip install -q \
    "agno==1.4.4" \
    "openai>=1.66.0" \
    "yfinance==0.2.54" \
    "pyautogen==0.2.35" \
    "mcp" \
    "python-dotenv==1.0.1" \
    "ipykernel==6.29.5"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 6.29.5 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
# Configure - note code updated here,
config_list = [{
    "model": "openai/gpt-oss-120b",
    "api_key": os.environ.get("GROQ_API_KEY"),
    "base_url": "https://api.groq.com/openai/v1",
    "price": [0, 0]
}]

In [ ]:
from autogen import ConversableAgent, AssistantAgent, UserProxyAgent
from autogen import initiate_chats
import pprint

agent = ConversableAgent(
    name="chatbot",
    llm_config={"config_list": config_list,"price": [0, 0]},
    human_input_mode="NEVER",
)

In [ ]:
reply = agent.generate_reply(
    messages=[{"content": "Tell me a joke.", "role": "user"}]
)
# note code updated here,
print(reply)

Why don’t scientists trust atoms anymore?

Because they *make up* everything! 😄


1. ConversableAgent - ConversableAgents are designed for structured conversations.(example - onboarding)
2. AssistantAgent - AssistantAgent are task oriented agents, often having expertise
3. UserProxyAgent - act as intermediaries between the user and the system. They relay user inputs

In [ ]:
# Define the RegistrationAgent for collecting user details
registration_agent = ConversableAgent(
    name="User Registration Agent",
    system_message='''You are a friendly registration assistant.
    Your task is to collect the user's full name, company name, and the product they are interested in.
    Do not ask for additional details. Respond with 'END_SESSION' once all details are obtained.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [ ]:
# Define the HelpDeskAgent for troubleshooting
help_desk_agent = AssistantAgent(
    name="Help Desk Agent",
    system_message='''You are a knowledgeable support assistant.
    Your role is to assist users in buying our product.
    Offer clear steps, share useful resources, and escalate if required.
    Respond with 'END_SESSION' once the issue is addressed.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [ ]:
# Define the SalesAgent for product recommendations
sales_agent = AssistantAgent(
    name="Sales Recommendation Agent",
    system_message='''You are a sales assistant.
    Based on user information, suggest the most suitable product and provide additional details.
    If the user expresses interest, proceed with gathering purchase intent details.
    Respond with 'END_SESSION' when user decides to proceed or not.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [ ]:
# Define the RelayAgent for communication handling
relay_agent = UserProxyAgent(
    name="Interaction Relay Agent",
    system_message='''You act as a bridge between users and specialized agents.
    You do not process requests yourself but ensure smooth communication.''',
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
)

In [ ]:
conversation_flow = [
    {
        "sender": registration_agent,
        "recipient": relay_agent,
        "message": "Hi! I'm here to help with your onboarding. Please provide your full name, company, and the product you're interested in.",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Extract user details as JSON: {'full_name': '', 'company': '', 'product': ''}",
        },
        "max_turns": 2,
        "clear_history": True
    },
    {
        "sender": relay_agent,
        "recipient": sales_agent,
        "message": "Based on user details, suggest a suitable product.",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Summarize recommended products as JSON: {'recommended_product': '', 'justification': ''}",
        },
        "max_turns": 3,
        "clear_history": False
    },
    {
        "sender": relay_agent,
        "recipient": help_desk_agent,
        "message": "Based on sale recommendation, closen the conversation with formal sale pitch",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Summarize the assistance provided as JSON: {'resolution_steps': []}",
        },
        "max_turns": 3,
        "clear_history": False
    }
]

# Execute the conversations
session_results = initiate_chats(conversation_flow)
pprint.pp(session_results)


********************************************************************************
Starting a new chat....

********************************************************************************
User Registration Agent (to Interaction Relay Agent):

Hi! I'm here to help with your onboarding. Please provide your full name, company, and the product you're interested in.

--------------------------------------------------------------------------------
Replying as Interaction Relay Agent. Provide feedback to User Registration Agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: Aman kumar
Interaction Relay Agent (to User Registration Agent):

Aman kumar

--------------------------------------------------------------------------------
User Registration Agent (to Interaction Relay Agent):

Thanks, Aman Kumar! Could you let me know the name of your company?

--------------------------------------------------------------------------------
Replying as Interaction Rela

/usr/local/lib/python3.12/dist-packages/autogen/agentchat/conversable_agent.py:1258: UserWarning: Cannot extract summary using reflection_with_llm: Error code: 400 - {'error': {'message': "Parsing failed. The model generated output that could not be parsed. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'output_parse_failed', 'failed_generation': ''}}. Using an empty str as summary.
  warnings.warn(


Sales Recommendation Agent (to Interaction Relay Agent):

Sure! To recommend the perfect product for you, I just need a few quick details:

1. **What type of product are you interested in?** (e.g., smartphone, laptop, home appliance, fitness gear, etc.)  
2. **What’s your primary use case or need?** (e.g., work, gaming, travel, home cooking, health tracking…)  
3. **Do you have a budget range in mind?**  
4. **Any specific features you’re looking for?** (e.g., long battery life, high‑resolution screen, waterproof, smart home integration…)

Let me know, and I’ll suggest the best match for you!

--------------------------------------------------------------------------------
Replying as Interaction Relay Agent. Provide feedback to Sales Recommendation Agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: 

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...
Interaction Relay Agent (to Sales Recommendation Agent):



---------------------------